# 🎨 LoRA Inference Pipeline

This notebook handles **only** the LoRA-based room generation:

1. **Input**: Wall/boundary images
2. **CNN Segmentation**: Extract room masks using DeepLabV3+
3. **Room Extraction**: Separate individual room regions
4. **LoRA Generation**: Generate furnished rooms per room type using FLUX.1-dev
5. **Output**: Save generated room images to folders

This follows the **same approach** as the individual inference notebooks (living_room, bedroom, etc.)

---

## 📦 Step 0: Setup & Imports

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set HuggingFace token
import os

def get_secret(name):
    """Read a secret from Colab Secrets (key icon in the sidebar), else from the environment."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ[name]

hf_token = get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("✓ HF_TOKEN environment variable set")

In [ ]:
# Install required packages
!pip install -q hf_transfer
!pip install -q diffusers transformers accelerate safetensors
!pip install -q opencv-python
!pip install -U controlnet-aux
print("✓ All packages installed")

In [ ]:
import os
import cv2
import glob
import numpy as np
import torch
import torchvision.transforms as T
from torchvision.models.segmentation import deeplabv3_resnet50
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from diffusers import FluxControlPipeline
from diffusers.utils import load_image
from controlnet_aux import CannyDetector

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 🔧 Configuration

In [ ]:
# ============================================================
# CONFIGURATION - Modify these paths as needed
# ============================================================

# Input: Wall images folder
WALL_IMAGES_DIR = '/content/drive/MyDrive/cnn_dataset/walls'

# CNN Segmentation model
CNN_MODEL_PATH = '/content/drive/MyDrive/cnn_dataset/best_model_final_4.pth'

# Output directories
SEGMENTATION_OUTPUT = '/content/drive/MyDrive/pipeline_output/segmentation'
EXTRACTION_OUTPUT = '/content/drive/MyDrive/pipeline_output/extraction'
LORA_OUTPUT = '/content/drive/MyDrive/pipeline_output/generated'

# LoRA weights paths (per room type) - these are FOLDERS containing safetensors
LORA_WEIGHTS = {
    'living_room': '/content/drive/MyDrive/outputs_livingroom_new',
    'bedroom': '/content/drive/MyDrive/outputs_bedroom_modified',
    'bathroom': '/content/drive/MyDrive/outputs_bathroom_modified',
    'hallway': '/content/drive/MyDrive/outputs_hallways',
    'toilet': '/content/drive/MyDrive/outputs_toilet'
}

# Create output directories
for output_dir in [SEGMENTATION_OUTPUT, EXTRACTION_OUTPUT, LORA_OUTPUT]:
    os.makedirs(output_dir, exist_ok=True)

print("✓ Configuration loaded")
print(f"  Wall images: {WALL_IMAGES_DIR}")
print(f"  CNN model: {CNN_MODEL_PATH}")

---
## 🧠 Part 1: CNN Segmentation

Load the trained DeepLabV3+ model and segment wall images into room masks.

In [ ]:
# Color map for segmentation classes
COLOR_MAP = np.array([
    [0, 0, 0],         # 0: background
    [255, 154, 0],     # 1: bedroom (orange)
    [50, 99, 155],     # 2: hallway (purple)
    [0, 155, 255],     # 3: bathroom (blue)
    [155, 255, 0],     # 4: living room (yellow-green)
    [100, 80, 71]      # 5: empty-room (brown)
], dtype=np.uint8)

CLASS_NAMES = ['background', 'bedroom', 'hallway', 'bathroom', 'living_room', 'empty_room']
NUM_CLASSES = len(CLASS_NAMES)

# Map class index to room type for LoRA
CLASS_TO_ROOM = {
    1: 'bedroom',
    2: 'hallway',
    3: 'bathroom',
    4: 'living_room',
    5: 'empty_room'  # Will use living_room LoRA as fallback
}

print(f"✓ {NUM_CLASSES} classes: {CLASS_NAMES}")

In [ ]:
def post_process(seg_map, min_area=500):
    """Remove small connected components from segmentation."""
    out = np.zeros_like(seg_map)
    unique_classes = np.unique(seg_map)

    for cls in unique_classes:
        mask = (seg_map == cls).astype(np.uint8)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)

        for label in range(1, num_labels):
            area = stats[label, cv2.CC_STAT_AREA]
            if area >= min_area:
                out[labels == label] = cls

    return out


def load_segmentation_model(model_path):
    """Load the trained DeepLabV3+ model."""
    model = deeplabv3_resnet50(pretrained_backbone=False, num_classes=NUM_CLASSES)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"✓ CNN model loaded from {model_path}")
    return model


def segment_image(model, image_path, img_size=(512, 512)):
    """
    Segment a wall image into room masks.
    
    Returns:
        seg_map: Segmentation map (H, W) with class indices
        color_overlay: RGB visualization of segmentation
        orig: Original image
    """
    # Load and preprocess
    orig = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    H0, W0 = orig.shape[:2]
    
    inp = cv2.resize(orig, img_size)
    inp_tensor = T.ToTensor()(inp).unsqueeze(0).to(device)
    
    # Inference
    with torch.no_grad():
        out = model(inp_tensor)['out'][0]
        prob = torch.softmax(out, dim=0).cpu().numpy()
        seg = prob.argmax(axis=0).astype(np.uint8)
        
        # Resize back to original size
        seg = cv2.resize(seg, (W0, H0), interpolation=cv2.INTER_NEAREST)
        seg = post_process(seg)
    
    # Create color overlay
    color_overlay = COLOR_MAP[seg]
    
    return seg, color_overlay, orig


# Load the model
cnn_model = load_segmentation_model(CNN_MODEL_PATH)

In [ ]:
def extract_room_masks(seg_map, original_image, output_dir, base_name):
    """
    Extract individual room masks from segmentation.
    
    Returns:
        room_masks: Dict mapping room_type -> list of mask info dicts
    """
    room_masks = {}
    
    for class_idx in range(1, NUM_CLASSES):  # Skip background
        class_mask = (seg_map == class_idx).astype(np.uint8)
        
        if class_mask.sum() == 0:
            continue
            
        room_type = CLASS_TO_ROOM.get(class_idx, 'unknown')
        
        # Find connected components (multiple instances of same room type)
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(class_mask)
        
        for label_idx in range(1, num_labels):  # Skip background label
            instance_mask = (labels == label_idx).astype(np.uint8) * 255
            
            # Get bounding box
            x, y, w, h, area = stats[label_idx]
            
            if area < 500:  # Skip tiny regions
                continue
            
            # Create room key with instance number
            if room_type not in room_masks:
                room_masks[room_type] = []
            
            instance_num = len(room_masks[room_type]) + 1
            mask_name = f"{base_name}_{room_type}_{instance_num}.png"
            mask_path = os.path.join(output_dir, room_type, mask_name)
            
            # Save mask and verify it was written
            os.makedirs(os.path.dirname(mask_path), exist_ok=True)
            success = cv2.imwrite(mask_path, instance_mask)
            
            # Verify file exists (important for Google Drive sync)
            if not success or not os.path.exists(mask_path):
                print(f"  ⚠ Failed to save mask: {mask_path}")
                continue
            
            room_masks[room_type].append({
                'path': mask_path,
                'mask': instance_mask,
                'bbox': (x, y, w, h),
                'area': area,
                'instance': instance_num
            })
    
    return room_masks


print("✓ Room extraction functions defined")

In [ ]:
# Test segmentation on a single image
test_images = glob.glob(os.path.join(WALL_IMAGES_DIR, '*.png')) + glob.glob(os.path.join(WALL_IMAGES_DIR, '*.jpg'))

if test_images:
    test_img = test_images[0]
    print(f"Testing with: {os.path.basename(test_img)}")
    
    seg_map, color_overlay, orig = segment_image(cnn_model, test_img)
    
    # Visualize segmentation
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(orig)
    axes[0].set_title('Original Wall Image')
    axes[0].axis('off')
    
    axes[1].imshow(color_overlay)
    axes[1].set_title('Segmentation')
    axes[1].axis('off')
    
    blend = ((0.6 * orig) + (0.4 * color_overlay)).astype(np.uint8)
    axes[2].imshow(blend)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Show detected rooms
    unique_classes = np.unique(seg_map)
    print(f"\nDetected rooms: {[CLASS_NAMES[c] for c in unique_classes if c > 0]}")
    
    # Extract room masks
    base_name = os.path.splitext(os.path.basename(test_img))[0]
    room_masks = extract_room_masks(seg_map, orig, EXTRACTION_OUTPUT, base_name)
    
    # ============================================================
    # VISUALIZE EACH ROOM MASK
    # ============================================================
    print(f"\n{'='*60}")
    print("🔍 ROOM MASKS VISUALIZATION")
    print(f"{'='*60}")
    
    total_masks = sum(len(masks) for masks in room_masks.values())
    print(f"Total room instances: {total_masks}\n")
    
    for room_type, mask_list in room_masks.items():
        print(f"\n📍 {room_type.upper()}: {len(mask_list)} instance(s)")
        
        # Create subplot for this room type
        n_instances = len(mask_list)
        fig, axes = plt.subplots(1, n_instances + 1, figsize=(5 * (n_instances + 1), 5))
        if n_instances == 0:
            continue
        if n_instances == 1:
            axes = [axes] if not isinstance(axes, np.ndarray) else axes
        
        # Show original image first
        axes[0].imshow(orig)
        axes[0].set_title(f'Original Image')
        axes[0].axis('off')
        
        for i, mask_info in enumerate(mask_list):
            ax = axes[i + 1]
            
            # Show mask
            mask = mask_info['mask']
            ax.imshow(mask, cmap='gray')
            
            # Add bbox info
            x, y, w, h = mask_info['bbox']
            area = mask_info['area']
            instance = mask_info['instance']
            
            ax.set_title(f'{room_type} #{instance}\nBBox: ({x},{y}) {w}x{h}\nArea: {area} px')
            ax.axis('off')
            
            print(f"  Instance #{instance}: bbox=({x},{y},{w},{h}), area={area}px")
        
        plt.tight_layout()
        plt.show()
    
    print(f"\n{'='*60}")
    print("✅ Masks look correct? If yes, proceed to LoRA generation!")
    print(f"{'='*60}")
    
else:
    print(f"No images found in {WALL_IMAGES_DIR}")

---
## 🎨 Part 2: LoRA Inference

Generate furnished room images using FLUX.1-dev with LoRA weights and Canny ControlNet.

**This follows the EXACT same approach as the individual inference notebooks.**

In [ ]:
# Room-specific prompts (same as individual notebooks)
ROOM_PROMPTS = {
    'living_room': "A top down 2d floor plan drawing of a living room. The room contains a rectangular dining table with chairs, a sofa, coffee table, and a TV mounted on the wall.",
    'bedroom': "A top down 2d floor plan drawing of a bedroom. The room contains a bed, nightstands, wardrobe, and desk.",
    'bathroom': "A top down 2d floor plan drawing of a bathroom. The room contains bathtub, shower stall, toilet, and sink.",
    'hallway': "A top down 2d floor plan drawing of a hallway with minimal furniture.",
    'toilet': "A top down 2d floor plan drawing of a small toilet room with toilet and sink.",
    'empty_room': "A top down 2d floor plan drawing of an empty room."
}

print("✓ Room prompts configured")

In [ ]:
# Helper functions from individual notebooks

def classify_shape(cnt):
    """Classify the shape of a contour as square, rectangle, or L-shaped."""
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
    x, y, w, h = cv2.boundingRect(approx)
    aspect_ratio = w / float(h)

    verts = len(approx)
    is_convex = cv2.isContourConvex(approx)

    if verts == 4 and is_convex:
        if 0.9 < aspect_ratio < 1.1:
            return "square"
        else:
            return "rectangle"
    else:
        return "L-shaped"


def classify_region(mask):
    """Classify the region of the mask within the image (top/bottom, left/right)."""
    h, w = mask.shape
    M = cv2.moments(mask)
    if M["m00"] == 0:
        return "center"
    cx = M["m10"] / M["m00"]
    cy = M["m01"] / M["m00"]

    vert = "top" if cy < h/2 else "bottom"
    horz = "left" if cx < w/2 else "right"
    return f"{vert} {horz}"


def create_binary_mask(mask_image):
    """Create a binary mask from the input mask image using contour fill method."""
    mask_image_np = np.array(mask_image)

    if len(mask_image_np.shape) == 3:
        mask_image_np = cv2.cvtColor(mask_image_np, cv2.COLOR_RGB2GRAY)

    kernel = np.ones((3, 3), np.uint8)
    closed = cv2.morphologyEx(mask_image_np, cv2.MORPH_CLOSE, kernel, iterations=3)

    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    binary_mask = np.zeros_like(mask_image_np)
    if contours:
        for contour in contours:
            area = cv2.contourArea(contour)
            if area > 100:
                cv2.drawContours(binary_mask, [contour], -1, 255, thickness=cv2.FILLED)

    binary_mask_final = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel, iterations=5)

    return binary_mask_final


def extract_room_from_generated(generated_image, binary_mask, output_path):
    """
    Extract the room part from generated image using the binary mask.
    Returns image with transparent background outside room.
    """
    # Convert PIL to numpy if needed
    if isinstance(generated_image, Image.Image):
        image = np.array(generated_image)
    else:
        image = generated_image.copy()
    
    # Resize mask to match image dimensions if needed
    if image.shape[:2] != binary_mask.shape[:2]:
        binary_mask = cv2.resize(binary_mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

    # Ensure mask is binary (0 or 255)
    _, mask_binary = cv2.threshold(binary_mask, 127, 255, cv2.THRESH_BINARY)

    # Convert image to RGBA
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGBA)
    elif image.shape[2] == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2RGBA)

    # Apply mask to create transparency
    image[:, :, 3] = mask_binary

    # Save
    extracted_img = Image.fromarray(image, 'RGBA')
    extracted_img.save(output_path)
    
    return extracted_img


print("✓ Helper functions defined")

In [ ]:
# Initialize Canny detector (same as individual notebooks)
canny_processor = CannyDetector()
print("✓ CannyDetector initialized")

In [ ]:
# Initialize the FLUX pipeline (same as individual notebooks)
print("Loading FLUX.1-dev pipeline...")
pipe = FluxControlPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.float16
)
pipe.to("cuda")
print("✓ Pipeline loaded")

In [ ]:
def load_lora_for_room(pipe, room_type, lora_weights_dict):
    """
    Load LoRA weights for a specific room type + Canny LoRA.
    This follows the EXACT same approach as individual inference notebooks.
    """
    # Get LoRA path for this room type
    lora_path = lora_weights_dict.get(room_type)
    if lora_path is None:
        lora_path = lora_weights_dict.get('living_room')
        print(f"  ⚠ No LoRA for {room_type}, using living_room")
    
    if not os.path.exists(lora_path):
        print(f"  ✗ LoRA path not found: {lora_path}")
        return False
    
    # Unload any existing LoRA weights
    try:
        pipe.unload_lora_weights()
    except:
        pass
    
    # Load room-specific LoRA
    pipe.load_lora_weights(lora_path, adapter_name="room")
    
    # Load Canny ControlNet LoRA (CRITICAL - this is what working notebooks use!)
    pipe.load_lora_weights(
        "black-forest-labs/FLUX.1-Canny-dev-lora",
        adapter_name="canny"
    )
    
    # Set adapter weights (same as individual notebooks)
    pipe.set_adapters(
        ["room", "canny"],
        adapter_weights=[0.8, 0.7]
    )
    
    # Enable memory optimization
    pipe.enable_model_cpu_offload()
    
    print(f"  ✓ Loaded LoRA: {room_type} + Canny adapter")
    return True


print("✓ LoRA loading function defined")

In [ ]:
def generate_room_image(pipe, mask_info, room_type, output_folder, base_name,
                        num_inference_steps=30, guidance_scale=7.5, num_images=1):
    """
    Generate furnished room image from a mask.
    This follows the EXACT same approach as individual inference notebooks.
    
    Args:
        pipe: FluxControlPipeline with LoRAs loaded
        mask_info: Dict containing mask data (path, mask array, bbox, etc.)
        room_type: Type of room (living_room, bedroom, etc.)
        output_folder: Base output folder
        base_name: Base name for output files
        num_inference_steps: Number of denoising steps
        guidance_scale: Guidance scale for generation
        num_images: Number of images to generate per mask
        
    Returns:
        List of generated image paths
    """
    mask_path = mask_info['path']
    instance_num = mask_info['instance']
    instance_mask = mask_info['mask']  # Use the mask array directly
    
    # Convert mask array to PIL Image (avoid file I/O issues)
    mask_image = Image.fromarray(instance_mask).convert('RGB')
    
    # Create binary mask for extraction later
    binary_mask = create_binary_mask(mask_image)
    
    # Analyze shape and region for better prompts
    _, thresh = cv2.threshold(instance_mask, 127, 255, cv2.THRESH_BINARY)
    cnts, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    shape = "rectangle"
    region = "center"
    if cnts:
        cnt = max(cnts, key=cv2.contourArea)
        shape = classify_shape(cnt)
        region = classify_region(thresh)
    
    # Process with Canny detector (SAME as individual notebooks!)
    control_image = canny_processor(
        mask_image,
        low_threshold=50,
        high_threshold=200,
        detect_resolution=512,
        image_resolution=512
    )
    
    # Build prompt with shape and region info
    base_prompt = ROOM_PROMPTS.get(room_type, ROOM_PROMPTS['empty_room'])
    prompt = f"A top down 2d floor plan drawing of a {shape} {room_type.replace('_', ' ')} situated on the {region} side. " + base_prompt.split('. ', 1)[-1]
    
    print(f"    Shape: {shape}, Region: {region}")
    print(f"    Prompt: {prompt[:80]}...")
    
    # Generate images (same parameters as individual notebooks)
    generator = torch.Generator(device="cuda").manual_seed(42)
    
    images = pipe(
        prompt=prompt,
        control_image=control_image,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
        num_images_per_prompt=num_images
    ).images
    
    # Save generated and extracted images
    output_paths = []
    
    # Create output directories
    generated_dir = os.path.join(output_folder, room_type, "generated")
    extracted_dir = os.path.join(output_folder, room_type, "extracted")
    os.makedirs(generated_dir, exist_ok=True)
    os.makedirs(extracted_dir, exist_ok=True)
    
    for i, img in enumerate(images):
        # Save raw generated image
        gen_name = f"{base_name}_{room_type}_{instance_num}_gen_{i+1}.png"
        gen_path = os.path.join(generated_dir, gen_name)
        img.save(gen_path)
        
        # Save extracted image (with mask applied)
        ext_name = f"{base_name}_{room_type}_{instance_num}_{shape}_{region.replace(' ', '_')}_extracted_{i+1}.png"
        ext_path = os.path.join(extracted_dir, ext_name)
        extract_room_from_generated(img, binary_mask, ext_path)
        
        output_paths.append({
            'generated': gen_path,
            'extracted': ext_path
        })
        print(f"    ✓ Saved: {gen_name}")
    
    return output_paths


print("✓ Room generation function defined")

---
## 🚀 Part 3: Run the Pipeline

In [ ]:
def run_lora_pipeline(wall_image_path, output_prefix):
    """
    Run the LoRA inference pipeline on a single wall image.
    
    Pipeline Steps:
    1. CNN Segmentation → Room masks
    2. Room Extraction → Individual room regions
    3. LoRA Generation → Furnished room images
    4. Save outputs to folders
    
    Args:
        wall_image_path: Path to input wall/boundary image
        output_prefix: Prefix for output files
        
    Returns:
        Dictionary with all generated room images
    """
    print(f"\n{'='*60}")
    print(f"🏠 LORA PIPELINE: {os.path.basename(wall_image_path)}")
    print(f"{'='*60}")
    
    results = {'input': wall_image_path, 'rooms': {}}
    
    # ---- Step 1: CNN Segmentation ----
    print("\n📍 Step 1: CNN Segmentation")
    seg_map, color_overlay, orig = segment_image(cnn_model, wall_image_path)
    
    # Save segmentation
    seg_path = os.path.join(SEGMENTATION_OUTPUT, f"{output_prefix}_segmentation.png")
    cv2.imwrite(seg_path, cv2.cvtColor(color_overlay, cv2.COLOR_RGB2BGR))
    results['segmentation_path'] = seg_path
    print(f"  ✓ Segmentation saved: {seg_path}")
    
    # Show detected rooms
    unique_classes = np.unique(seg_map)
    detected_rooms = [CLASS_NAMES[c] for c in unique_classes if c > 0]
    print(f"  ✓ Detected rooms: {detected_rooms}")
    
    # ---- Step 2: Extract Room Masks ----
    print("\n📍 Step 2: Extracting Room Masks")
    room_masks = extract_room_masks(seg_map, orig, EXTRACTION_OUTPUT, output_prefix)
    
    for room_type, masks in room_masks.items():
        print(f"  ✓ {room_type}: {len(masks)} instance(s)")
    
    # ---- Step 3: LoRA Generation ----
    print("\n📍 Step 3: LoRA Room Generation")
    
    current_lora_room = None
    
    for room_type, mask_list in room_masks.items():
        results['rooms'][room_type] = []
        
        # Load LoRA for this room type (only if different from current)
        lora_room = room_type if room_type != 'empty_room' else 'living_room'
        if current_lora_room != lora_room:
            print(f"\n  Loading LoRA for: {lora_room}")
            if not load_lora_for_room(pipe, lora_room, LORA_WEIGHTS):
                print(f"  ✗ Failed to load LoRA for {lora_room}, skipping...")
                continue
            current_lora_room = lora_room
        
        for mask_info in mask_list:
            instance = mask_info['instance']
            
            print(f"\n  Generating {room_type} #{instance}...")
            
            try:
                output_paths = generate_room_image(
                    pipe=pipe,
                    mask_info=mask_info,  # Pass entire mask_info dict
                    room_type=room_type,
                    output_folder=LORA_OUTPUT,
                    base_name=output_prefix,
                    num_images=4  # Generate 4 images per room (same as individual notebooks)
                )
                
                results['rooms'][room_type].append({
                    'instance': instance,
                    'mask_path': mask_info['path'],
                    'bbox': mask_info['bbox'],
                    'outputs': output_paths
                })
                
            except Exception as e:
                print(f"    ✗ Failed: {e}")
                import traceback
                traceback.print_exc()
    
    print(f"\n{'='*60}")
    print("✅ LORA PIPELINE COMPLETE!")
    print(f"{'='*60}")
    
    # Summary
    total_generated = sum(len(rooms) for rooms in results['rooms'].values())
    print(f"\n📊 Summary:")
    print(f"  - Rooms detected: {len(room_masks)}")
    print(f"  - Images generated: {total_generated}")
    print(f"  - Output folder: {LORA_OUTPUT}")
    
    return results


print("✓ Pipeline function defined")

In [ ]:
# ============================================================
# RUN THE PIPELINE ON A SINGLE IMAGE
# ============================================================

# Get list of wall images
wall_images = glob.glob(os.path.join(WALL_IMAGES_DIR, '*.png')) + glob.glob(os.path.join(WALL_IMAGES_DIR, '*.jpg'))
print(f"Found {len(wall_images)} wall images")

if wall_images:
    # Process first image as demo
    test_image = wall_images[0]
    base_name = os.path.splitext(os.path.basename(test_image))[0]
    
    results = run_lora_pipeline(test_image, base_name)
    
    # Visualize some results
    print("\n🖼️ Visualizing results...")
    
    for room_type, room_list in results['rooms'].items():
        for room_info in room_list:
            if room_info['outputs']:
                gen_path = room_info['outputs'][0]['generated']
                ext_path = room_info['outputs'][0]['extracted']
                
                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
                
                # Original mask
                mask_img = cv2.imread(room_info['mask_path'], cv2.IMREAD_GRAYSCALE)
                axes[0].imshow(mask_img, cmap='gray')
                axes[0].set_title(f'Mask: {room_type} #{room_info["instance"]}')
                axes[0].axis('off')
                
                # Generated image
                gen_img = Image.open(gen_path)
                axes[1].imshow(gen_img)
                axes[1].set_title('Generated')
                axes[1].axis('off')
                
                # Extracted image
                ext_img = Image.open(ext_path)
                axes[2].imshow(ext_img)
                axes[2].set_title('Extracted (with mask)')
                axes[2].axis('off')
                
                plt.tight_layout()
                plt.show()
else:
    print(f"No wall images found in {WALL_IMAGES_DIR}")

---
## 📊 Batch Processing (Optional)

In [ ]:
def run_batch_lora_pipeline(wall_images_dir, max_images=None):
    """
    Process multiple wall images through the LoRA pipeline.
    
    Args:
        wall_images_dir: Directory containing wall images
        max_images: Maximum number of images to process (None = all)
        
    Returns:
        List of result dictionaries
    """
    wall_images = glob.glob(os.path.join(wall_images_dir, '*.png')) + glob.glob(os.path.join(wall_images_dir, '*.jpg'))
    
    if max_images:
        wall_images = wall_images[:max_images]
    
    print(f"\n🚀 Processing {len(wall_images)} images...\n")
    
    all_results = []
    for i, img_path in enumerate(tqdm(wall_images, desc="Processing")):
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        output_prefix = f"{i+1:03d}_{base_name}"
        
        try:
            results = run_lora_pipeline(img_path, output_prefix)
            all_results.append(results)
        except Exception as e:
            print(f"\n✗ Failed on {img_path}: {e}")
            continue
    
    print(f"\n\n{'='*60}")
    print(f"✅ Batch processing complete: {len(all_results)}/{len(wall_images)} successful")
    print(f"{'='*60}")
    
    return all_results


print("✓ Batch processing function defined")

In [ ]:
# ============================================================
# 🚀 RUN BATCH PROCESSING
# ============================================================
# Uncomment ONE of the options below:

# Option 1: Process first N images
# all_results = run_batch_lora_pipeline(WALL_IMAGES_DIR, max_images=5)

# Option 2: Process ALL images in the folder
# all_results = run_batch_lora_pipeline(WALL_IMAGES_DIR)

# Option 3: Process specific images by index range
# wall_images = glob.glob(os.path.join(WALL_IMAGES_DIR, '*.png')) + glob.glob(os.path.join(WALL_IMAGES_DIR, '*.jpg'))
# for i, img_path in enumerate(wall_images[0:10]):  # Images 0-9
#     base_name = os.path.splitext(os.path.basename(img_path))[0]
#     results = run_lora_pipeline(img_path, f"{i+1:05d}_{base_name}")

# ============================================================
# EXAMPLE: Process 50 images
# ============================================================
all_results = run_batch_lora_pipeline(WALL_IMAGES_DIR, max_images=50)

# Show summary
print(f"\n📊 BATCH COMPLETE!")
print(f"Total images processed: {len(all_results)}")
total_rooms = sum(sum(len(r) for r in res['rooms'].values()) for res in all_results)
print(f"Total rooms generated: {total_rooms}")
print(f"Output saved to: {LORA_OUTPUT}")

---
## 📈 Pipeline Summary

This notebook implements the **LoRA-only inference pipeline**:

| Stage | Component | Description |
|-------|-----------|-------------|
| 1 | **CNN Segmentation** | DeepLabV3+ extracts room masks from wall images |
| 2 | **Room Extraction** | Separates individual room regions with bounding boxes |
| 3 | **LoRA Generation** | FLUX.1-dev + LoRA + Canny ControlNet generates furnished room images |
| 4 | **Output Saving** | Saves generated and extracted images to organized folders |

### Output Structure:
```
pipeline_output/
├── segmentation/
│   └── {image}_segmentation.png
├── extraction/
│   ├── living_room/
│   ├── bedroom/
│   └── ...
└── generated/
    ├── living_room/
    │   ├── generated/
    │   │   └── {image}_living_room_1_gen_1.png
    │   └── extracted/
    │       └── {image}_living_room_1_rectangle_top_left_extracted_1.png
    ├── bedroom/
    └── ...
```

### Key Features (matching individual notebooks):
- ✅ Uses `CannyDetector` from `controlnet_aux` (proper edge detection)
- ✅ Loads `FLUX.1-Canny-dev-lora` adapter (required for ControlNet)
- ✅ Uses `set_adapters` with weights [0.8, 0.7]
- ✅ Shape/region classification for better prompts
- ✅ Extracts room with transparency mask
- ✅ Same inference parameters (30 steps, 7.5 guidance)